# Module 9b: Charts and the calculations behind them

**Goal:** know, for every chart type, **what is calculated** and **what it means**, so you can choose the right chart, set its options deliberately, defend it in a thesis, and spot when a chart misleads.

**You will learn:** histograms (binning rules, density) · the ECDF · kernel density estimates · box plots (quartiles, fences, whiskers, notches) · error bars (SD vs SE vs CI vs bootstrap) ·
rolling averages and EWMA · regression lines with **confidence vs prediction bands** · correlation heatmaps · 2-D density · proportion charts

**Maths in this module:** quantiles · bin-width rules · the empirical CDF · kernel smoothing · standard errors and intervals · leverage in regression bands · rank correlation

> 🧮 **A chart is a calculation, drawn.** Whenever you call a plotting function, ask: *what numbers did it compute from my data, and are those the numbers I want to show?*
> Below, every chart's calculation is done **by hand first**, then checked against the library.

| chart | what is calculated |
|---|---|
| histogram | counts (or densities) of values falling in each bin |
| ECDF | proportion of values $\le x$ at each $x$ |
| density (KDE) | an average of small bumps centred on each data point |
| box plot | median, quartiles, fences, whisker ends, outliers |
| bar/dot with error bars | a statistic **and** a measure of spread or uncertainty |
| line (rolling/EWMA) | a moving average over time |
| scatter + trend | a fitted line with a confidence (or prediction) band |
| heatmap | a matrix of correlations (or counts) mapped to colour |
| stacked / proportion bars | counts or proportions per category (with intervals) |

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

sns.set_theme(style="whitegrid", palette="colorblind")
students = pd.read_csv(DATA / "students.csv")
rides = pd.read_csv(DATA / "daily_rides.csv", parse_dates=["date"])
desk = pd.read_csv(DATA / "service_desk.csv")
trial = pd.read_csv(DATA / "app_trial.csv")

## 1. Histogram: counting values in bins

Split the range into **bins** (intervals) and count how many values fall in each. Two choices matter: the **bin width** and whether the height is a **count** or a **density**
(count divided by $n\times\text{width}$, so that the total **area is 1** and histograms with different bin widths or sample sizes are comparable, and can be overlaid with a fitted curve).

In [ ]:
scores = students["exam_score"].to_numpy()

def hist_by_hand(x, edges):
    counts = np.array([((x >= lo) & (x < hi)).sum() for lo, hi in zip(edges[:-1], edges[1:])])
    counts[-1] += (x == edges[-1]).sum()              # the last bin includes its right edge
    return counts

edges = np.linspace(scores.min(), scores.max(), 11)    # 10 equal-width bins
mine = hist_by_hand(scores, edges)
numpy_counts, _ = np.histogram(scores, bins=edges)
print("by hand:", mine)
print("numpy  :", numpy_counts, "-> identical:", (mine == numpy_counts).all())

width = edges[1] - edges[0]
density = mine / (len(scores) * width)
print("area under the density histogram =", round((density * width).sum(), 6))

### How many bins? Three standard rules

| rule | number of bins / width | idea |
|---|---|---|
| **Sturges** | $k=\lceil\log_2 n\rceil+1$ | fine for small, roughly normal data; too few bins for large $n$ |
| **Scott** | width $=3.49\,s\,n^{-1/3}$ | minimises error for normal data |
| **Freedman–Diaconis** | width $=2\,\text{IQR}\,n^{-1/3}$ | uses the IQR, so robust to outliers and skew |

In [ ]:
def bin_rules(x):
    x = np.asarray(x, dtype=float)
    n = len(x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    rng_ = x.max() - x.min()
    return {
        "sturges": (rng_ / (math.ceil(math.log2(n)) + 1), math.ceil(math.log2(n)) + 1),
        "scott": (3.49 * x.std(ddof=1) * n ** (-1 / 3), None),
        "freedman-diaconis": (2 * iqr * n ** (-1 / 3), None),
    }

resp = desk["response_minutes"].to_numpy()
for name, data in {"exam scores (symmetric)": scores, "response minutes (right-skewed)": resp}.items():
    print(name)
    for rule, (w, k) in bin_rules(data).items():
        k = k or math.ceil((data.max() - data.min()) / w)
        numpy_name = "fd" if rule == "freedman-diaconis" else rule
        print(f"   {rule:18s} width {w:6.2f} -> {k:3d} bins   (numpy agrees: {len(np.histogram_bin_edges(data, numpy_name)) - 1} bins)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.3))
for ax, bins in zip(axes, [4, "fd", 80]):
    ax.hist(resp, bins=bins, density=True, edgecolor="white")
    ax.set_title(f"bins = {bins}" + (" (Freedman-Diaconis)" if bins == "fd" else ""))
    ax.set_xlabel("Response minutes")
axes[0].set_ylabel("Density")
plt.tight_layout(); plt.show()

**Pitfalls:** too few bins hide structure (the bimodality in `call_minutes` disappears); too many show noise. Always state whether the $y$-axis is **count or density**, and don't use unequal bin widths with count heights.

## 2. The empirical CDF (ECDF): no binning at all

$$\hat F(x)=\frac{\#\{x_i\le x\}}{n}$$

The ECDF is a staircase that rises $1/n$ at each data point. It needs **no bin or bandwidth choice**, shows the whole distribution, lets you **read percentiles directly** (where the curve crosses 0.5 is the median),
and makes **comparing groups** easy: the largest vertical gap between two ECDFs is the **Kolmogorov–Smirnov distance**.

In [ ]:
def ecdf(x):
    xs = np.sort(np.asarray(x, dtype=float))
    return xs, np.arange(1, len(xs) + 1) / len(xs)

mins = {g: trial.loc[trial["group"] == g, "minutes_active"].to_numpy() for g in ("control", "gamified")}
fig, ax = plt.subplots(figsize=(7, 3.8))
for g, v in mins.items():
    xs, ys = ecdf(v)
    ax.step(xs, ys, where="post", label=g)

a, b = mins["control"], mins["gamified"]
grid = np.sort(np.r_[a, b])
Fa, Fb = np.searchsorted(np.sort(a), grid, side="right") / len(a), np.searchsorted(np.sort(b), grid, side="right") / len(b)
gap = np.abs(Fa - Fb); at = grid[gap.argmax()]
ax.vlines(at, min(Fa[gap.argmax()], Fb[gap.argmax()]), max(Fa[gap.argmax()], Fb[gap.argmax()]), color="k", linestyle="--")
ax.set_xlim(0, 120); ax.set_xlabel("Minutes active"); ax.set_ylabel("Proportion of users at or below"); ax.legend()
ax.set_title(f"Largest gap between the ECDFs (KS distance) = {gap.max():.2f}")
plt.show()
print("KS distance by hand", round(gap.max(), 4), "| scipy ks_2samp", round(stats.ks_2samp(a, b).statistic, 4))
xs_b, ys_b = ecdf(b)
print(f"median read off the ECDF (first x where it reaches 0.5): {xs_b[np.argmax(ys_b >= 0.5)]:.1f}  | np.median averages the two middle values: {np.median(b):.1f}")

## 3. Kernel density estimate (KDE): a smooth histogram

Place a small bump (a **kernel**, usually a normal curve) on every data point and average them:

$$\hat f(x)=\frac1{nh}\sum_{i=1}^{n}K\!\Big(\frac{x-x_i}{h}\Big),\qquad K(u)=\frac{1}{\sqrt{2\pi}}e^{-u^2/2}$$

The **bandwidth $h$** controls smoothness (like bin width). A common default (Scott's rule, used by SciPy and seaborn): $h=s\,n^{-1/5}$.

In [ ]:
def kde_by_hand(x, points, h=None):
    x = np.asarray(x, dtype=float)
    h = h if h is not None else x.std(ddof=1) * len(x) ** (-1 / 5)         # Scott's rule
    u = (np.asarray(points)[:, None] - x[None, :]) / h
    return (np.exp(-u ** 2 / 2) / math.sqrt(2 * math.pi)).mean(axis=1) / h

pts = np.array([55.0, 68.0, 80.0])
print("by hand:", kde_by_hand(scores, pts).round(5))
print("scipy  :", stats.gaussian_kde(scores)(pts).round(5))

h0 = scores.std(ddof=1) * len(scores) ** (-1 / 5)
xs = np.linspace(30, 100, 300)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(scores, bins=20, density=True, alpha=0.3, color="grey")
for mult, label in [(0.3, "h/3.3: undersmoothed (wiggly)"), (1, "h = Scott's rule"), (3, "3h: oversmoothed (blurry)")]:
    ax.plot(xs, kde_by_hand(scores, xs, h0 * mult), label=label)
ax.set_xlabel("Exam score"); ax.set_ylabel("Density"); ax.legend(fontsize=8); plt.show()
print(f"Scott bandwidth h = {h0:.2f} points")

**KDE pitfalls:** for **bounded or positive** data (times, scores capped at 100) the smooth curve leaks past the boundary (negative minutes!); use `cut=0` in seaborn, a log scale, or report an ECDF instead.
With small $n$ a KDE can invent bumps. **Violin plots** are a KDE mirrored left–right (usually with a box plot inside).

## 4. Box plot: five numbers and a rule

The box spans the **quartiles** $Q_1$ to $Q_3$ with a line at the median. With $\text{IQR}=Q_3-Q_1$, the **fences** are $Q_1-1.5\,\text{IQR}$ and $Q_3+1.5\,\text{IQR}$.
The **whiskers** extend to the **most extreme data points that are still inside the fences** (not to the fences themselves), and anything beyond is plotted as an **outlier** point.
The optional **notch** is an approximate 95% interval for the median: $\tilde x\pm1.57\,\text{IQR}/\sqrt n$ (if two notches don't overlap, the medians likely differ).

In [ ]:
def box_stats(x):
    x = np.sort(np.asarray(x, dtype=float))
    q1, med, q3 = np.percentile(x, [25, 50, 75])
    iqr = q3 - q1
    lo_fence, hi_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    inside = x[(x >= lo_fence) & (x <= hi_fence)]
    return {"q1": q1, "med": med, "q3": q3, "whislo": inside.min(), "whishi": inside.max(),
            "fliers": x[(x < lo_fence) | (x > hi_fence)],
            "cilo": med - 1.57 * iqr / math.sqrt(len(x)), "cihi": med + 1.57 * iqr / math.sqrt(len(x))}

mine = box_stats(resp)
ref = cbook.boxplot_stats(resp, whis=1.5)[0]
for key in ("q1", "med", "q3", "whislo", "whishi", "cilo", "cihi"):
    assert abs(mine[key] - ref[key]) < 1e-9, key
assert np.allclose(mine["fliers"], np.sort(ref["fliers"]))      # matplotlib lists outliers in the data's original order
assert (mine["fliers"] > mine["whishi"]).all(), "every outlier should lie above the upper whisker"
print({k: round(v, 2) for k, v in mine.items() if k != "fliers"}, "| outliers:", len(mine["fliers"]), "(all above the upper fence, a right-skewed variable)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].boxplot(resp, notch=True); axes[0].set_ylabel("Response minutes"); axes[0].set_title("box plot of response minutes (notched)")
sns.violinplot(data=trial, x="group", y="minutes_active", cut=0, ax=axes[1]); axes[1].set_title("violin = KDE mirrored (cut=0)")
plt.tight_layout(); plt.show()

A box plot summarises **five numbers**, so it hides multimodality (a bimodal variable can look the same as a unimodal one); for small groups, overlay the points (`sns.stripplot`).

## 5. Bars and dots with error bars: which interval?

"Mean $\pm$ error bar" is the most misused chart in student work. Four *different* quantities all get drawn as error bars:

| error bar | formula | what it tells you |
|---|---|---|
| **SD** | $s$ | the **spread of the data** (not uncertainty about the mean) |
| **SE** | $s/\sqrt n$ | the **uncertainty of the mean** (roughly a 68% interval) |
| **95% CI** | $\bar x\pm t^*\,s/\sqrt n$ | a range of plausible values for the population mean |
| **bootstrap CI** | percentiles of resampled means | like the CI, no normality assumed |

**Always say in the caption which one you plotted**, and prefer the 95% CI (or bootstrap) for comparing groups. For proportions use the **Wilson** interval.

In [ ]:
def error_bars(x, rng=np.random.default_rng(0)):
    x = np.asarray(x, dtype=float)
    n = len(x)
    se = x.std(ddof=1) / math.sqrt(n)
    boot = rng.choice(x, size=(4000, n)).mean(axis=1)
    return {"mean": x.mean(), "SD": x.std(ddof=1), "SE": se,
            "CI95": stats.t.ppf(0.975, n - 1) * se, "boot_lo": np.quantile(boot, 0.025), "boot_hi": np.quantile(boot, 0.975)}

rows = {g: error_bars(trial.loc[trial["group"] == g, "post_score"]) for g in ("control", "tutorial", "gamified")}
print(pd.DataFrame(rows).T.round(2))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6), sharey=True)
for ax, (label, key) in zip(axes, [("mean ± SD", "SD"), ("mean ± SE", "SE"), ("mean ± 95% CI", "CI95")]):
    for i, (g, r) in enumerate(rows.items()):
        ax.errorbar(i, r["mean"], yerr=r[key], fmt="o", capsize=6, markersize=8)
    ax.set_xticks(range(3)); ax.set_xticklabels(rows.keys()); ax.set_title(label)
axes[0].set_ylabel("Post score")
plt.tight_layout(); plt.show()

The **same data** produce three very different pictures: with SD bars everything overlaps (people vary a lot), with SE bars the groups look separated. Neither is "wrong", but they answer different questions,
which is why the caption must say which. (Overlapping 95% CIs of two groups do **not** automatically mean "no significant difference", so test the *difference* and show its interval.)

**Proportions:** a bar's height $\hat p=x/n$ needs an interval too. The Wilson interval is the right default:

In [ ]:
conv = trial.groupby("group")["converted"].agg(["sum", "count"])
conv["rate"] = conv["sum"] / conv["count"]
ci = np.array([proportion_confint(s, n, method="wilson") for s, n in zip(conv["sum"], conv["count"])])
conv["lo"], conv["hi"] = ci[:, 0], ci[:, 1]
print(conv.round(3))

fig, ax = plt.subplots(figsize=(5.5, 3.4))
ax.errorbar(conv["rate"], range(3), xerr=[conv["rate"] - conv["lo"], conv["hi"] - conv["rate"]], fmt="o", capsize=5)
ax.set_yticks(range(3)); ax.set_yticklabels(conv.index); ax.set_xlabel("Conversion rate (Wilson 95% CI)"); ax.set_xlim(0, 0.6)
plt.show()

## 6. Line charts over time: smoothing is a calculation

Daily data are noisy; two standard smoothers:

* **Moving (rolling) average** over a window of $w$ points: $\bar y_t=\frac1w\sum_{j=0}^{w-1}y_{t-j}$ (trailing). Wider window = smoother, but lags the data.
* **Exponentially weighted moving average (EWMA):** $s_t=\alpha y_t+(1-\alpha)s_{t-1}$, with $s_0=y_0$; recent points count most. Larger $\alpha$ = less smoothing.

In [ ]:
y = rides.set_index("date")["rides"]

def rolling_by_hand(v, w):
    v = np.asarray(v, dtype=float)
    out = np.full(len(v), np.nan)
    for t in range(w - 1, len(v)):
        out[t] = v[t - w + 1:t + 1].mean()
    return out

def ewma_by_hand(v, alpha):
    v = np.asarray(v, dtype=float)
    s = np.empty(len(v)); s[0] = v[0]
    for t in range(1, len(v)):
        s[t] = alpha * v[t] + (1 - alpha) * s[t - 1]
    return s

print("rolling matches pandas:", np.allclose(rolling_by_hand(y, 7)[6:], y.rolling(7).mean().to_numpy()[6:]))
print("EWMA matches pandas   :", np.allclose(ewma_by_hand(y, 0.2), y.ewm(alpha=0.2, adjust=False).mean().to_numpy()))

fig, ax = plt.subplots(figsize=(10, 3.6))
ax.plot(y.index, y, color="lightgrey", label="daily")
ax.plot(y.index, rolling_by_hand(y, 7), label="7-day rolling mean")
ax.plot(y.index, rolling_by_hand(y, 30), label="30-day rolling mean")
ax.plot(y.index, ewma_by_hand(y, 0.1), label="EWMA, alpha = 0.1")
ax.set_ylabel("Rides per day"); ax.legend(ncol=4, fontsize=8); plt.show()

Rolling averages **lag** turning points (look at the 30-day line behind the July peak). Report the window or $\alpha$. For a year-on-year or month-on-month comparison, indexing to a base (100 at the start) or showing **percent change** (Module 5) is often clearer.

## 7. Scatter with a trend line: confidence band vs prediction band

A fitted line $\hat y=b_0+b_1x$ comes with two different bands, and mixing them up is a classic thesis error:

* **Confidence band** for the *mean* response at $x_0$: $\hat y_0\pm t^*\,s\sqrt{\dfrac1n+\dfrac{(x_0-\bar x)^2}{S_{xx}}}$. It narrows with more data and is **narrowest at $\bar x$**.
* **Prediction band** for a *new individual* at $x_0$: $\hat y_0\pm t^*\,s\sqrt{1+\dfrac1n+\dfrac{(x_0-\bar x)^2}{S_{xx}}}$. It stays wide, however much data you have, because individuals scatter around the line.

Here $s^2=\text{SSE}/(n-2)$, $S_{xx}=\sum(x_i-\bar x)^2$ and $t^*$ has $n-2$ degrees of freedom.

In [ ]:
def regression_band(x, y, x0, level=0.95):
    x, y, x0 = (np.asarray(v, dtype=float) for v in (x, y, x0))
    n = len(x)
    b1, b0 = np.polyfit(x, y, 1)
    s = math.sqrt(((y - (b0 + b1 * x)) ** 2).sum() / (n - 2))
    sxx = ((x - x.mean()) ** 2).sum()
    lever = 1 / n + (x0 - x.mean()) ** 2 / sxx
    t_star = stats.t.ppf((1 + level) / 2, n - 2)
    fit = b0 + b1 * x0
    return fit, fit - t_star * s * np.sqrt(lever), fit + t_star * s * np.sqrt(lever), fit - t_star * s * np.sqrt(1 + lever), fit + t_star * s * np.sqrt(1 + lever)

xs, ys = students["study_hours_week"], students["exam_score"]
grid = np.linspace(xs.min(), xs.max(), 100)
fit, c_lo, c_hi, p_lo, p_hi = regression_band(xs, ys, grid)

# check against statsmodels
sm_fit = smf.ols("exam_score ~ study_hours_week", students).fit().get_prediction(pd.DataFrame({"study_hours_week": grid})).summary_frame(alpha=0.05)
print("confidence band matches statsmodels:", np.allclose(c_lo, sm_fit["mean_ci_lower"]) and np.allclose(c_hi, sm_fit["mean_ci_upper"]))
print("prediction band matches statsmodels:", np.allclose(p_lo, sm_fit["obs_ci_lower"]) and np.allclose(p_hi, sm_fit["obs_ci_upper"]))

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(xs, ys, alpha=0.4, s=15)
ax.plot(grid, fit, color="black")
ax.fill_between(grid, c_lo, c_hi, alpha=0.5, label="95% confidence band (mean score)")
ax.fill_between(grid, p_lo, p_hi, alpha=0.2, label="95% prediction band (a new student)")
ax.set_xlabel("Study hours per week"); ax.set_ylabel("Exam score"); ax.legend(fontsize=8)
plt.show()
print(f"at 10 hours: mean score {regression_band(xs, ys, [10])[0][0]:.1f}; the mean is known to within ±{(regression_band(xs, ys, [10])[2][0] - regression_band(xs, ys, [10])[0][0]):.1f}, but a single student could score ±{(regression_band(xs, ys, [10])[4][0] - regression_band(xs, ys, [10])[0][0]):.1f} from it")

Notice that the prediction band rises above 100, which is impossible for an exam score capped at 100. A straight-line model is only a **local approximation**; bands can't know about ceilings, so say so (or model a bounded outcome properly).

Seaborn's `regplot` draws only the **confidence band** (computed by bootstrap). **LOWESS** (`statsmodels.nonparametric.lowess`) fits a *local* smooth curve that needn't be straight, a useful check on linearity. Don't use a smoother to make a claim about a straight-line relationship.

## 8. Correlation heatmap: a matrix of calculations

Each cell is a correlation $r_{jk}$. By hand: **standardise** each column ($z$-scores, Module 3), then $R=\dfrac{1}{n-1}Z^\top Z$. **Spearman** correlation is the same calculation on **ranks**.

In [ ]:
cols = ["study_hours_week", "sleep_hours", "attendance_pct", "exam_score"]
Z = ((students[cols] - students[cols].mean()) / students[cols].std(ddof=1)).to_numpy()
R = Z.T @ Z / (len(Z) - 1)
ranks = students[cols].rank().to_numpy()
Zr = (ranks - ranks.mean(axis=0)) / ranks.std(axis=0, ddof=1)
Rs = Zr.T @ Zr / (len(Zr) - 1)
print("Pearson matches pandas :", np.allclose(R, students[cols].corr().to_numpy()))
print("Spearman matches pandas:", np.allclose(Rs, students[cols].corr(method="spearman").to_numpy()))

mask = np.triu(np.ones_like(R, dtype=bool), k=1)             # hide the redundant upper triangle
fig, ax = plt.subplots(figsize=(5.5, 4.2))
sns.heatmap(pd.DataFrame(R, index=cols, columns=cols), mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title("Pearson correlations")
plt.show()

**Heatmap rules:** use a **diverging** palette centred on 0 with fixed limits $[-1,1]$; hide the redundant triangle; annotate values; and remember the matrix shows only *linear* association between pairs.
With many variables, 5% of correlations will look "significant" by chance, so see Module 8b on multiple comparisons.

## 9. Two numbers, many points: 2-D binning and density

A scatter plot with thousands of points becomes a blob (**overplotting**). Count points in 2-D bins (a **hexbin / 2-D histogram**) or estimate a 2-D density:

In [ ]:
def hist2d_by_hand(x, y, xedges, yedges):
    counts = np.zeros((len(xedges) - 1, len(yedges) - 1), dtype=int)
    for xi, yi in zip(x, y):
        i = min(np.searchsorted(xedges, xi, side="right") - 1, len(xedges) - 2)
        j = min(np.searchsorted(yedges, yi, side="right") - 1, len(yedges) - 2)
        counts[i, j] += 1
    return counts

xe, ye = np.linspace(rides["temp_c"].min(), rides["temp_c"].max(), 9), np.linspace(rides["rides"].min(), rides["rides"].max(), 9)
valid = rides.dropna(subset=["temp_c"])
mine = hist2d_by_hand(valid["temp_c"], valid["rides"], xe, ye)
ref, _, _ = np.histogram2d(valid["temp_c"], valid["rides"], bins=[xe, ye])
print("2-D counts match numpy:", (mine == ref).all(), "| total counted:", mine.sum())

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].hexbin(valid["temp_c"], valid["rides"], gridsize=15, cmap="Blues", mincnt=1); axes[0].set_title("hexbin: points per cell")
sns.kdeplot(data=valid, x="temp_c", y="rides", fill=True, levels=8, cmap="Blues", ax=axes[1]); axes[1].set_title("2-D density contours")
for ax in axes: ax.set_xlabel("Temperature (°C)"); ax.set_ylabel("Rides")
plt.tight_layout(); plt.show()

## 10. Categories: counts, proportions and stacked bars

The calculation is a **cross-tabulation**, then (usually) **normalising** so rows sum to 100% (compare *composition*) or columns do.

In [ ]:
counts = pd.crosstab(trial["group"], trial["device"])
row_props = counts.div(counts.sum(axis=1), axis=0)
print(counts, "\n"); print((row_props * 100).round(1))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
counts.plot.bar(stacked=True, ax=axes[0], rot=0); axes[0].set_title("stacked counts: hard to compare shares")
(row_props * 100).plot.bar(stacked=True, ax=axes[1], rot=0); axes[1].set_title("100% stacked: compares composition")
axes[0].set_ylabel("Users"); axes[1].set_ylabel("% of group")
plt.tight_layout(); plt.show()

## 11. Putting it together: the chart-and-calculation table

| chart | calculation | key choice | typical mistake |
|---|---|---|---|
| histogram | bin counts / density | bin width (Sturges, Scott, FD) | too few/many bins; count vs density unlabelled |
| ECDF | $\#\{x_i\le x\}/n$ | none | not used when comparing groups |
| KDE / violin | kernel average | bandwidth $h$ | density leaking past a boundary |
| box plot | $Q_1$, median, $Q_3$, fences, whiskers | whisker rule | assuming whiskers are min/max; hiding bimodality |
| bar + error bar | mean ± (SD, SE, CI) | **which** interval | unlabelled error bars; SE vs SD confusion; bars not from zero |
| proportion chart | $x/n$ + Wilson CI | interval method | no interval for small $n$ |
| rolling / EWMA | moving average | window $w$ or $\alpha$ | forgetting the lag; unreported window |
| scatter + fit | regression line + band | confidence vs prediction | calling a confidence band a "range of the data" |
| heatmap | correlation matrix | method; colour scale | sequential palette for ±; no fixed limits |
| hexbin / 2-D KDE | 2-D counts / density | bin size / bandwidth | overplotting |
| stacked bars | cross-tab (normalised?) | counts vs % | comparing non-baseline segments |

A one-page version, with more chart types, is in [`reference/choosing-a-chart.md`](../../reference/choosing-a-chart.md).

### A thesis-grade figure caption
> **Figure 3.** Post-test scores by onboarding group (n = 100 per group). Points show group means; error bars are **95% confidence intervals** (t-based). The dashed line marks no change. Source: randomised trial, 2024.

State *what* is plotted, *the sample size*, *what the bars/bands mean*, and *the source*.

## Recap

* Every chart is a calculation: know it, and choose its parameters (bins, bandwidth, interval type, window) deliberately and **report them**.
* Prefer **ECDFs** for comparing distributions, **intervals** (not bare bars) for comparing estimates, **confidence vs prediction bands** appropriately, and **diverging scales** for correlations.
* Label error bars; start bars at zero; show the data when $n$ is small.

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the **project checkpoint** (a chart audit).